In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp, lit

file_path = "/Volumes/workspace/stock_data/api_keys/fmp_api_key.txt"

with open(file_path, "r") as f:
    api_key = f.read().strip()

tickers = [
    'AAPL', 'TSLA', 'AMZN', 'MSFT', 'NVDA', 'GOOGL', 'META', 'NFLX', 'JPM', 'V', 
    'BAC', 'AMD', 'PYPL', 'DIS', 'T', 'PFE', 'COST', 'INTC', 'KO', 'TGT', 'NKE', 
    'SPY', 'BA', 'BABA', 'XOM', 'WMT', 'GE', 'CSCO', 'VZ', 'JNJ', 'CVX', 'PLTR', 
    'SQ', 'SHOP', 'SBUX', 'SOFI', 'HOOD', 'RBLX', 'SNAP', 'UBER', 'FDX', 'ABBV', 
    'ETSY', 'MRNA', 'LMT', 'GM', 'F', 'RIVN', 'LCID', 'CCL', 'DAL', 'UAL', 'AAL', 
    'TSM', 'SONY', 'ET', 'NOK', 'MRO', 'COIN', 'SIRI', 'RIOT', 'CPRX', 'VWO', 'SPYG', 
    'ROKU', 'VIAC', 'ATVI', 'BIDU', 'DOCU', 'ZM', 'PINS', 'TLRY', 'WBA', 'MGM', 
    'NIO', 'C', 'GS', 'WFC', 'ADBE', 'PEP', 'UNH', 'CARR', 'FUBO', 'HCA', 'TWTR', 
    'BILI', 'RKT'
]

def get_company_profile_raw(ticker, api_key):
    url = "https://financialmodelingprep.com/stable/profile"
    params = {
        "symbol": ticker, 
        "apikey": api_key
    }
    response = requests.get(url, params=params)
    return (ticker, response.text, "fmp_profile_api")

bronze_schema = StructType([
    StructField("ticker", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("source", StringType(), True),
    StructField("ingest_timestamp", TimestampType(), True),
])

data = [(*get_company_profile_raw(ticker, api_key), None) for ticker in tickers]

bronze_df = spark.createDataFrame(data, schema=bronze_schema).withColumn("ingest_timestamp", current_timestamp())

bronze_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.company_profiles_bronze')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.company_profiles_bronze